# Notebook 15 — Cluster-Robust Village Bootstrap

ASER is a two-stage cluster sample: villages/enumeration blocks within districts, then ~5% of households within each village. Resampling children independently (notebook 08) treats within-village children as independent and produces CIs that are too narrow. This notebook resamples **whole villages** with replacement (cluster bootstrap) and compares results to the naive child-level bootstrap.

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Override with ASER_CHILD_XLSX environment variable if set.
XLSX_PATH  = os.environ.get('ASER_CHILD_XLSX', '../data/raw/ITAASER2023Child.xlsx')
DATA_PATH  = "../Data:/child_merged.csv"
DS_PATH    = "../Data:/district_summary.csv"
OUT_PATH   = Path("../outputs")
OUT_PATH.mkdir(exist_ok=True)

N_BOOT = 5000
SEED   = 42
print(f"Setup complete.  XLSX_PATH = {XLSX_PATH}")

## Step 1 — Attach VCODES to child_merged.csv

VCODES is absent from `child_merged.csv` (confirmed). Re-derive by loading `Id` and `VCODES` from the raw ASER Excel file and merging on `Id = child_id`. Merge match rate is reported before proceeding.

In [ ]:
# ── 1a: confirm absence ──
cm_cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
print("VCODES in child_merged.csv:", "VCODES" in cm_cols)
print("Columns:", cm_cols)
print()

# ── 1b: load xlsx (Id + VCODES only) ──
print("Loading VCODES from xlsx …")
raw_xl = pd.read_excel(XLSX_PATH, usecols=["Id", "VCODES"])
raw_xl.columns = ["child_id", "VCODES"]
print(f"xlsx rows: {len(raw_xl):,}  |  unique VCODES: {raw_xl['VCODES'].nunique():,}")
print(f"VCODES dtype: {raw_xl['VCODES'].dtype}  |  any null: {raw_xl['VCODES'].isna().sum()}")

# ── 1c: load child_merged ──
print()
print("Loading child_merged.csv …")
usecols = ["child_id", "district", "province", "school_type", "arithmetic_score"]
cm = pd.read_csv(DATA_PATH, usecols=usecols)
print(f"child_merged rows: {len(cm):,}")

# ── 1d: merge and report match rate ──
merged = cm.merge(raw_xl, on="child_id", how="left")
n_matched = merged["VCODES"].notna().sum()
n_unmatched = merged["VCODES"].isna().sum()
print()
print(f"After left-merge on child_id:")
print(f"  Matched (VCODES present): {n_matched:,}  ({n_matched/len(merged):.2%})")
print(f"  Unmatched (VCODES null):  {n_unmatched:,}  ({n_unmatched/len(merged):.2%})")
if n_unmatched > 0:
    print("  Unmatched sample:")
    print(merged[merged["VCODES"].isna()].head(5)[["child_id","district"]].to_string())

# ── 1e: village count per district ──
print()
vill_per_dist = merged.groupby("district")["VCODES"].nunique().sort_values()
print("Villages per district (after merge):")
print(vill_per_dist.describe().round(1))
print()
print("Districts with fewest villages:")
print(vill_per_dist.head(10).to_string())

## Step 2 — Bootstrap Functions

**Child-level bootstrap** (from notebook 08): resamples individual government and private children independently with replacement.

**Cluster bootstrap**: for each of 5,000 iterations, draws the district's villages with replacement (size = n_villages in that district), pools all children from drawn villages, then computes the gap. The vectorised implementation uses `np.bincount` to assign per-child weights equal to how many times their village was drawn, then computes weighted means — equivalent to concatenating drawn-village children but ~10× faster.

In [ ]:
def child_bootstrap_gap_ci(govt_scores, pvt_scores, n_boot=N_BOOT, seed=SEED):
    """Naive child-level bootstrap (replicates notebook 08 exactly)."""
    g = np.asarray(govt_scores, dtype=float)
    p = np.asarray(pvt_scores,  dtype=float)
    if len(g) < 2 or len(p) < 2:
        return np.nan, np.nan
    rng   = np.random.default_rng(seed)
    idx_g = rng.integers(0, len(g), (n_boot, len(g)))
    idx_p = rng.integers(0, len(p), (n_boot, len(p)))
    gaps  = p[idx_p].mean(axis=1) - g[idx_g].mean(axis=1)
    return float(np.percentile(gaps, 2.5)), float(np.percentile(gaps, 97.5))


def cluster_bootstrap_gap_ci(df_district, n_boot=N_BOOT, seed=SEED):
    """
    Cluster bootstrap: resample WHOLE villages with replacement.

    Vectorised via np.bincount: each child gets weight = number of times
    its village was drawn, then weighted means are computed with dot products.
    This is algebraically identical to concatenating children from drawn
    villages but avoids Python-level array building inside the loop.
    """
    villages  = df_district["VCODES"].values
    unique_v  = np.unique(villages[~pd.isnull(villages)])
    n_v       = len(unique_v)

    if n_v < 2:
        return np.nan, np.nan   # cannot bootstrap from a single village

    # Map village codes to contiguous 0-based integers
    vill_map = {v: i for i, v in enumerate(unique_v)}
    vill_int = np.array([vill_map.get(v, -1) for v in villages])

    govt_mask  = (df_district["school_type"].values == "Government")
    pvt_mask   = (df_district["school_type"].values == "Private")
    scores     = df_district["arithmetic_score"].values.astype(float)
    valid      = ~np.isnan(scores)

    valid_govt  = (govt_mask & valid).astype(float)
    valid_pvt   = (pvt_mask  & valid).astype(float)
    scores_safe = np.where(valid, scores, 0.0)

    # Only children with a mapped village contribute
    in_scope = vill_int >= 0
    vill_int    = vill_int[in_scope]
    valid_govt  = valid_govt[in_scope]
    valid_pvt   = valid_pvt[in_scope]
    scores_safe = scores_safe[in_scope]

    rng = np.random.default_rng(seed)
    boot_gaps = []

    for _ in range(n_boot):
        drawn  = rng.integers(0, n_v, n_v)          # draw n_v villages w/ replacement
        counts = np.bincount(drawn, minlength=n_v)   # how many times each village drawn
        w      = counts[vill_int]                    # child-level weights

        wg = w * valid_govt
        wp = w * valid_pvt
        sg = wg.sum()
        sp = wp.sum()

        if sg > 0 and sp > 0:
            boot_gaps.append(
                np.dot(wp, scores_safe) / sp -
                np.dot(wg, scores_safe) / sg
            )

    if len(boot_gaps) < 100:
        return np.nan, np.nan
    return float(np.percentile(boot_gaps, 2.5)), float(np.percentile(boot_gaps, 97.5))


print("Bootstrap functions defined.")
print(f"  child_bootstrap_gap_ci : resamples children independently")
print(f"  cluster_bootstrap_gap_ci: resamples villages with replacement (cluster-robust)")

## Step 3 — Target Districts

Three sets:
1. **Top-10**: highest private advantage (from notebook 08)
2. **Bottom-15**: largest government advantage (from notebook 08)
3. **All filtered**: districts with n_govt ≥ 100 AND n_pvt ≥ 50 (n = 102)

In [ ]:
TOP10 = [
    "Barkhan", "Tando Muhammad Khan", "Kurram Agency", "Tank", "Kohlu",
    "South Waziristan Agency", "Sanghar", "Dadu", "Harnai", "Orakzai Agency",
]
BOTTOM15 = [
    "Faisalabad", "Sahiwal", "Sheikhupura", "Okara", "Diamer", "Lodhran",
    "Haripur", "Muzaffargarh", "Multan", "Kachhi", "Mirpur", "Sialkot",
    "Chakwal", "Dera Ghazi Khan", "Jhang",
]

ds = pd.read_csv(DS_PATH)
filtered = ds[
    (ds["n_government"] >= 100) &
    (ds["n_private"]    >= 50)  &
    (ds["arithmetic_gap"].notna())
]["district"].tolist()

print(f"Top-10 districts:    {len(TOP10)}")
print(f"Bottom-15 districts: {len(BOTTOM15)}")
print(f"Filtered districts:  {len(filtered)}")

# Check that top10/bottom15 are all within filtered (for consistent comparison)
not_in_filt = [d for d in TOP10 + BOTTOM15 if d not in filtered]
label = not_in_filt if not_in_filt else "none"
print(f"Top/bottom districts not in filtered sample: {label}")

## Step 4 — Run Both Bootstraps on All Target Districts

For each district in the combined set (filtered sample, which includes all top-10 and bottom-15), run both the child-level and cluster-level bootstrap. Report n_villages (unique VCODES within district) alongside gap and CI bounds.

In [ ]:
import time

# Work with all districts in the filtered sample (superset of top10+bottom15)
all_target = sorted(set(filtered + TOP10 + BOTTOM15))
print(f"Total districts to process: {len(all_target)}")
print()

# Filter merged df to Government/Private children with valid arithmetic score
child_df = merged[
    merged["school_type"].isin(["Government", "Private"]) &
    merged["arithmetic_score"].notna() &
    merged["VCODES"].notna()
].copy()

rows = []
t0 = time.time()

for i, dist in enumerate(sorted(all_target), 1):
    sub = child_df[child_df["district"] == dist]
    gov = sub[sub["school_type"] == "Government"]["arithmetic_score"].values
    pvt = sub[sub["school_type"] == "Private"]["arithmetic_score"].values

    if len(gov) < 2 or len(pvt) < 2:
        continue

    gap       = float(pvt.mean() - gov.mean())
    n_vills   = sub["VCODES"].nunique()

    # ── child-level bootstrap ──
    cl, ch = child_bootstrap_gap_ci(gov, pvt)

    # ── cluster bootstrap ──
    kl, kh = cluster_bootstrap_gap_ci(sub)

    rows.append({
        "district":         dist,
        "list":             ("top10"    if dist in TOP10 else
                             "bottom15" if dist in BOTTOM15 else "filtered"),
        "n_govt":           len(gov),
        "n_pvt":            len(pvt),
        "n_villages":       n_vills,
        "gap":              round(gap, 4),
        "child_CI_low":     round(cl,  4) if not np.isnan(cl) else np.nan,
        "child_CI_high":    round(ch,  4) if not np.isnan(ch) else np.nan,
        "cluster_CI_low":   round(kl,  4) if not np.isnan(kl) else np.nan,
        "cluster_CI_high":  round(kh,  4) if not np.isnan(kh) else np.nan,
        "child_excl_zero":  (not np.isnan(cl)) and not (cl <= 0 <= ch),
        "cluster_excl_zero":(not np.isnan(kl)) and not (kl <= 0 <= kh),
    })

    if i % 10 == 0 or i == len(all_target):
        elapsed = time.time() - t0
        print(f"  {i}/{len(all_target)} done  ({elapsed:.1f}s elapsed)")

result_df = pd.DataFrame(rows)
print(f"\nDone. {len(result_df)} districts processed.")

## Step 5 — Comparison Tables and Summary

In [ ]:
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

cols_show = ["district","n_govt","n_pvt","n_villages","gap",
             "child_CI_low","child_CI_high","cluster_CI_low","cluster_CI_high",
             "child_excl_zero","cluster_excl_zero"]

top10_res    = result_df[result_df["list"] == "top10"].sort_values("gap", ascending=False)
bottom15_res = result_df[result_df["list"] == "bottom15"].sort_values("gap")

print("=" * 110)
print("TOP-10 DISTRICTS (highest private advantage)")
print("=" * 110)
print(top10_res[cols_show].to_string(index=False))

print()
print("=" * 110)
print("BOTTOM-15 DISTRICTS (largest government advantage)")
print("=" * 110)
print(bottom15_res[cols_show].to_string(index=False))

In [ ]:
print("=" * 110)
print("FULL FILTERED SAMPLE (n_govt>=100, n_pvt>=50)")
print("=" * 110)
all_res = result_df.sort_values("gap", ascending=False)
print(all_res[cols_show].to_string(index=False))

In [ ]:
# ── Direction-change summary ──
print("=" * 70)
print("SUMMARY: how cluster bootstrap changes conclusions")
print("=" * 70)
print()

total = len(result_df)
changed = result_df[result_df["child_excl_zero"] != result_df["cluster_excl_zero"]]
print(f"Total districts analysed:  {total}")
print(f"Districts where conclusion CHANGES (CI crosses zero in one but not the other): {len(changed)}")
if len(changed):
    print()
    print("Changed districts:")
    print(changed[["district","list","gap","child_CI_low","child_CI_high",
                   "cluster_CI_low","cluster_CI_high","n_villages"]].to_string(index=False))

print()
print("── TOP-10 check ──")
t10 = result_df[result_df["list"] == "top10"]
print(f"All 10 child CIs above zero:   {t10['child_excl_zero'].all() and (t10['gap']>0).all()}")
print(f"All 10 cluster CIs above zero: {t10['cluster_excl_zero'].all() and (t10['gap']>0).all()}")
t10_fail = t10[~t10["cluster_excl_zero"]]
if len(t10_fail):
    print(f"Top-10 districts where cluster CI crosses zero: {t10_fail['district'].tolist()}")

print()
print("── BOTTOM-15 check ──")
b15 = result_df[result_df["list"] == "bottom15"]
print(f"All 15 child CIs below zero:   {b15['child_excl_zero'].all() and (b15['gap']<0).all()}")
print(f"All 15 cluster CIs below zero: {b15['cluster_excl_zero'].all() and (b15['gap']<0).all()}")
b15_fail = b15[~b15["cluster_excl_zero"]]
if len(b15_fail):
    print(f"Bottom-15 districts where cluster CI crosses zero: {b15_fail['district'].tolist()}")

print()
print("── CI width comparison (cluster / child ratio) ──")
result_df["child_width"]   = result_df["child_CI_high"]  - result_df["child_CI_low"]
result_df["cluster_width"] = result_df["cluster_CI_high"] - result_df["cluster_CI_low"]
result_df["width_ratio"]   = result_df["cluster_width"]  / result_df["child_width"]
valid_ratio = result_df["width_ratio"].dropna()
print(f"Median cluster/child CI width ratio: {valid_ratio.median():.2f}x")
print(f"Mean   cluster/child CI width ratio: {valid_ratio.mean():.2f}x")
print(f"Max    cluster/child CI width ratio: {valid_ratio.max():.2f}x")
print(f"Districts where cluster CI is NARROWER than child CI: {(valid_ratio < 1).sum()}")

narrower = result_df[result_df["width_ratio"].notna() & (result_df["width_ratio"] < 1)]
if len(narrower) > 0:
    print("Narrower-CI districts:")
    for _, row in narrower.iterrows():
        print(f"  {row['district']}: n_villages={int(row['n_villages'])}, "
              f"width ratio={row['width_ratio']:.2f}x "
              f"(child {row['child_width']:.3f}, cluster {row['cluster_width']:.3f})")

print()
min_vill_idx = result_df["n_villages"].idxmin()
min_vill_n   = int(result_df.loc[min_vill_idx, "n_villages"])
min_vill_d   = result_df.loc[min_vill_idx, "district"]
print(f"Min villages among the 102 filtered districts: {min_vill_n} ({min_vill_d})")
print("Note: Bhakkar (6 villages in the raw survey) fails the n_pvt>=50 filter "
      "and is absent from this analysis.")

In [ ]:
save_cols = ["district","list","n_govt","n_pvt","n_villages","gap",
             "child_CI_low","child_CI_high","cluster_CI_low","cluster_CI_high",
             "child_excl_zero","cluster_excl_zero","child_width","cluster_width","width_ratio"]
result_df[save_cols].to_csv(OUT_PATH / "cluster_bootstrap_comparison.csv", index=False)
print(f"Saved {len(result_df)} rows → outputs/cluster_bootstrap_comparison.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# RECONCILED TABLES — canonical gap from district_summary.csv
# ══════════════════════════════════════════════════════════════════
# The bootstrap point estimates in result_df were computed from a
# filtered/cleaned child_merged slice and differ slightly from the
# pre-aggregated district_summary.csv values.  For reporting, use
# arithmetic_gap from district_summary.csv (canonical source) as
# the point estimate, and retain cluster CI bounds as-is.
# ══════════════════════════════════════════════════════════════════

ds_full = pd.read_csv(DS_PATH)

# Merge canonical gap and province into result_df
recon = result_df.merge(
    ds_full[["district", "province", "arithmetic_gap"]].rename(
        columns={"arithmetic_gap": "canonical_gap"}
    ),
    on="district", how="left"
)

# cluster_excl_zero is unchanged (property of the CI, not the point estimate)
OUT_COLS = ["district", "province", "canonical_gap",
            "child_CI_low", "child_CI_high",
            "cluster_CI_low", "cluster_CI_high",
            "cluster_excl_zero"]

# ── Top-10 ── sorted by canonical_gap descending
top10_recon = (
    recon[recon["list"] == "top10"]
    .sort_values("canonical_gap", ascending=False)
    [OUT_COLS]
    .reset_index(drop=True)
)

# ── Bottom-15 ── sorted by canonical_gap ascending (most-negative first)
bot15_recon = (
    recon[recon["list"] == "bottom15"]
    .sort_values("canonical_gap", ascending=True)
    [OUT_COLS]
    .reset_index(drop=True)
)

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 160)

print("=" * 130)
print("TOP-10 DISTRICTS — canonical gap from district_summary.csv, cluster CIs from bootstrap")
print("=" * 130)
print(top10_recon.to_string(index=False))

print()
print("=" * 130)
print("BOTTOM-15 DISTRICTS — canonical gap from district_summary.csv, cluster CIs from bootstrap")
print("=" * 130)
print(bot15_recon.to_string(index=False))

# ── Save ──
top10_recon.to_csv(OUT_PATH / "top10_reconciled.csv",   index=False)
bot15_recon.to_csv(OUT_PATH / "bottom15_reconciled.csv", index=False)
print(f"\nSaved → outputs/top10_reconciled.csv   ({len(top10_recon)} rows)")
print(f"Saved → outputs/bottom15_reconciled.csv ({len(bot15_recon)} rows)")